# NFTリスティングの高速化

NFTの一覧を取得するときのアクセスの高速化の手法についてサンプルを示します。

## 準備

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

動作確認に使用するユーザをロードします。

In [2]:
var users = [];
for(var i=0; i<5; i++){
    var wfstr = await loadWallet(`wallet-user${i}.json`);
    var wallet = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
    var id = (await rpc.call(wallet, 'c1query', { type: 'search', key: 'me' })).value[0].id;
    users[i] = { id, wallet };
    console.log(`user${i}:`, id, wallet.address);
}
var userids = users.map(u => u.id);

user0: u73621973 eQiA3bsfZunQ8KsRpkTgMGndwu46rq
user1: u28169743 epGyEm4BXFxFvLmXkDdDvcqspESPFu
user2: u53371386 eaZMspgRpi68EdReTZEiUkdJuKsdct
user3: u58185320 e5fVm5R4eKYXa7U57pshS4UQwuvGsp
user4: u10676071 eEYRycGm4znXxnf7bTngX72JmYk57r


## NFTコレクションのデプロイ

NFTコントラクトのサンプルコードを読み出します。

In [3]:
var { default: func } = await import('./nft100.mjs');
var m = /function.*\{([\s\S]*)\}/.exec(func.toString());
var code = m[1];

NFTコレクションの名称と略号を書き換えます。

In [4]:
var code = code.replace(/NFT_NAME = '.*'/, `NFT_NAME = 'sample NFT collection #123'`);
var code = code.replace(/NFT_SYMBOL = '.*'/, `NFT_SYMBOL = 'SMP123'`);

NFTコレクションの発行数を10とします。  
初期状態ではユーザ0が全トークンのオーナになります。

In [5]:
var code = code.replace(/TOTAL_SUPPLY = \d+/, `TOTAL_SUPPLY = 10`);
var code = code.replace(/INITIAL_OWNER = '.*'/, `INITIAL_OWNER = '${users[0].id}'`);

スマートコントラクトをデプロイします。  
変数nftidにNFTコントラクトのIDが格納されます。

In [6]:
var func = new Function(code);
var nftid = await deploySmartContract({func:'string', args:'any'}, func, { name: 'nft100' });

NFTコントラクトのアクセス範囲を、動作確認に使用するユーザに設定します。

In [7]:
await rpc.call(adminWallet, 'c1update', {id:nftid, prop:'accessible_to', value: userids});

{
  txno: 702285,
  txid: 'xqU49ECwrWfFKYqmXYdh54pgSg5vyM3fV3jnaPgUzbeyr',
  status: 'ok',
  value: null
}

## トークンオーナーの変更

所有者の取得結果が正しいことを後で確認しやすいように、トークンごとの所有者を変えます。

In [8]:
var wallet = users[0].wallet;
for(var i=0; i<10; i++){
    var userNo = i % 5;
    var from = users[0].id;
    var to = users[userNo].id;
    var tokenId = 'token' + i;
    var resp = await rpc.call(wallet, nftid, {func: 'transferFrom', args: { from, to, tokenId }});
    console.log(`transfer ${tokenId} to ${to}:`, resp.status);    
}

transfer token0 to u73621973: ok
transfer token1 to u28169743: ok
transfer token2 to u53371386: ok
transfer token3 to u58185320: ok
transfer token4 to u10676071: ok
transfer token5 to u73621973: ok
transfer token6 to u28169743: ok
transfer token7 to u53371386: ok
transfer token8 to u58185320: ok
transfer token9 to u10676071: ok


## トークンのリスティング（高速化なし）

ユーザ0がトークンをリスティングします。高速化の手段は講じていません。

In [9]:
var startTime = performance.now();
var wallet = users[0].wallet;

// トークンの個数を取得する。
var resp = await rpc.call(wallet, nftid, {func: 'totalSupply'});
var totalSupply = resp.value;

// トークンの各々について
for(var index = 0; index < totalSupply; index ++) {
    // 表示するトークンのIDを取得する。
    var resp = await rpc.call(wallet, nftid, {func: 'tokenByIndex', args: { index }});
    var tokenId = resp.value;
    // 表示するトークンのオーナーを取得する。
    var resp = await rpc.call(wallet, nftid, {func: 'ownerOf', args: { tokenId }});
    var owner = resp.value;
    // トークンのURIを取得する。
    var resp = await rpc.call(wallet, nftid, {func: 'tokenURI', args: { tokenId }});
    var tokenURI = resp.value;
    // トークンの情報をコンソールに出力する。
    console.log(`${tokenId} owner=${owner} tokenURI=${tokenURI}`);
}

console.log(`処理時間: ${Math.floor(performance.now() - startTime)/1000} 秒`);

token0 owner=u73621973 tokenURI=https://anywhere.com/token0
token1 owner=u28169743 tokenURI=https://anywhere.com/token1
token2 owner=u53371386 tokenURI=https://anywhere.com/token2
token3 owner=u58185320 tokenURI=https://anywhere.com/token3
token4 owner=u10676071 tokenURI=https://anywhere.com/token4
token5 owner=u73621973 tokenURI=https://anywhere.com/token5
token6 owner=u28169743 tokenURI=https://anywhere.com/token6
token7 owner=u53371386 tokenURI=https://anywhere.com/token7
token8 owner=u58185320 tokenURI=https://anywhere.com/token8
token9 owner=u10676071 tokenURI=https://anywhere.com/token9
処理時間: 11.113 秒


## readmodeを指定して高速化

readmode:'fast' を指定することで、書き込みのオーバヘッドを排除して高速化したバージョンです。  
※ 古いデータを取得しない目的で、接続先ピアの同期を待ち合わせるため、最初のリクエストだけ readmode:'full' としています。

In [10]:
var startTime = performance.now();
var wallet = users[0].wallet;

// トークンの個数を取得する。
var resp = await rpc.call(wallet, nftid, {func: 'totalSupply'}, {readmode: 'full'});
var totalSupply = resp.value;

for(var index = 0; index < totalSupply; index ++) {
    // 表示するトークンのIDを取得する。
    var resp = await rpc.call(wallet, nftid, {func: 'tokenByIndex', args: { index }}, {readmode: 'fast'});
    var tokenId = resp.value;
    // 表示するトークンのオーナーを取得する。
    var resp = await rpc.call(wallet, nftid, {func: 'ownerOf', args: { tokenId }}, {readmode: 'fast'});
    var owner = resp.value;
    // トークンのURIを取得する。
    var resp = await rpc.call(wallet, nftid, {func: 'tokenURI', args: { tokenId }}, {readmode: 'fast'});
    var tokenURI = resp.value;
    // トークンの情報をコンソールに出力する。
    console.log(`${tokenId} owner=${owner} tokenURI=${tokenURI}`);
}

console.log(`処理時間: ${Math.floor(performance.now() - startTime)/1000} 秒`);

token0 owner=u73621973 tokenURI=https://anywhere.com/token0
token1 owner=u28169743 tokenURI=https://anywhere.com/token1
token2 owner=u53371386 tokenURI=https://anywhere.com/token2
token3 owner=u58185320 tokenURI=https://anywhere.com/token3
token4 owner=u10676071 tokenURI=https://anywhere.com/token4
token5 owner=u73621973 tokenURI=https://anywhere.com/token5
token6 owner=u28169743 tokenURI=https://anywhere.com/token6
token7 owner=u53371386 tokenURI=https://anywhere.com/token7
token8 owner=u58185320 tokenURI=https://anywhere.com/token8
token9 owner=u10676071 tokenURI=https://anywhere.com/token9
処理時間: 3.864 秒


## 並列化による高速化

さらにトークンごとの情報取得を並列化することで、高速化したバージョンです。

In [11]:
var startTime = performance.now();
var wallet = users[0].wallet;

// トークンの個数を取得する。
var resp = await rpc.call(wallet, nftid, {func: 'totalSupply'}, {readmode: 'full'});
var totalSupply = resp.value;

var promises = [];
for(let index = 0; index < totalSupply; index ++) {
    promises.push(async function () {
        // 表示するトークンのIDを取得する。
        var resp = await rpc.call(wallet, nftid, {func: 'tokenByIndex', args: { index }}, {readmode: 'fast'});
        var tokenId = resp.value;
        // 表示するトークンのオーナーを取得する。
        var resp = await rpc.call(wallet, nftid, {func: 'ownerOf', args: { tokenId }}, {readmode: 'fast'});
        var owner = resp.value;
        // トークンのURIを取得する。
        var resp = await rpc.call(wallet, nftid, {func: 'tokenURI', args: { tokenId }}, {readmode: 'fast'});
        var tokenURI = resp.value;
        // トークンの情報を出力する。
        return `${tokenId} owner=${owner} tokenURI=${tokenURI}`;
    }());
}
var outputs = await Promise.all(promises);
console.log(outputs.join('\n'));

console.log(`処理時間: ${Math.floor(performance.now() - startTime)/1000} 秒`);

token0 owner=u73621973 tokenURI=https://anywhere.com/token0
token1 owner=u28169743 tokenURI=https://anywhere.com/token1
token2 owner=u53371386 tokenURI=https://anywhere.com/token2
token3 owner=u58185320 tokenURI=https://anywhere.com/token3
token4 owner=u10676071 tokenURI=https://anywhere.com/token4
token5 owner=u73621973 tokenURI=https://anywhere.com/token5
token6 owner=u28169743 tokenURI=https://anywhere.com/token6
token7 owner=u53371386 tokenURI=https://anywhere.com/token7
token8 owner=u58185320 tokenURI=https://anywhere.com/token8
token9 owner=u10676071 tokenURI=https://anywhere.com/token9
処理時間: 1.131 秒


## リスティング用コントラクトを用いた高速化

あらかじめリスティングのためのコントラクトをデプロイしておきます。

In [12]:
var func = function () {
    var nftContract = openContract(nftid, {delegation: true});
    var tokens = [];
    for(var index = start; index < end; index ++) {
        var tokenId = nftContract.call({func: 'tokenByIndex', args: {index: index}});
        var owner = nftContract.call({func: 'ownerOf', args: {tokenId: tokenId}});
        var tokenURI = nftContract.call({func : 'tokenURI', args: {tokenId: tokenId}});
        tokens.push( tokenId + ' owner=' + owner+ ' tokenURI=' + tokenURI);
    }
    return tokens;    
}
var nftListerId = await deploySmartContract({nftid: 'string', start: 'number', end: 'number'}, func, { name: 'nftLister' });
await rpc.call(adminWallet, 'c1update', {id:nftListerId, prop:'accessible_to', value: userids});

{
  txno: 702333,
  txid: 'x8dXmpEA8bBCYdv8rStHnmGg5FUmgHudCqyudYNwMrFwEB',
  status: 'ok',
  value: null
}

上でデプロイしたコントラクトを用いて、トークンのリスティングを高速に行います。  
※ 一回のスマートコントラクトの呼び出しで最大100個のトークンの情報を取得します。

In [13]:
var startTime = performance.now();
var wallet = users[0].wallet;

// トークンの個数を取得する。
var resp = await rpc.call(wallet, nftid, {func: 'totalSupply'}, {readmode: 'full'});
var totalSupply = resp.value;

for(var start = 0; start < totalSupply; ) {
    var end = Math.min(totalSupply, start + 100);
    var resp = await rpc.call(wallet, nftListerId, {nftid, start, end}, {readmode: 'fast'});
    console.log(resp.value.join('\n'));
    start = end;
}

console.log(`処理時間: ${Math.floor(performance.now() - startTime)/1000} 秒`);

token0 owner=u73621973 tokenURI=https://anywhere.com/token0
token1 owner=u28169743 tokenURI=https://anywhere.com/token1
token2 owner=u53371386 tokenURI=https://anywhere.com/token2
token3 owner=u58185320 tokenURI=https://anywhere.com/token3
token4 owner=u10676071 tokenURI=https://anywhere.com/token4
token5 owner=u73621973 tokenURI=https://anywhere.com/token5
token6 owner=u28169743 tokenURI=https://anywhere.com/token6
token7 owner=u53371386 tokenURI=https://anywhere.com/token7
token8 owner=u58185320 tokenURI=https://anywhere.com/token8
token9 owner=u10676071 tokenURI=https://anywhere.com/token9
処理時間: 0.604 秒
